In [ ]:
# !pip install  huggingface_hub openai tapipy

In [ ]:
import getpass
from tapipy.tapis import Tapis
from pathlib import Path
import time
import json

# call OpenAI API to test if the inference server is working.
from openai import OpenAI
import logging
from __future__ import annotations
from dataclasses import dataclass, field
from typing import List, Dict, Any
import httpx

default_logging_level = logging.CRITICAL
# default_logging_level = logging.DEBUG

# Silence almost everything (only critical errors)
logging.basicConfig(level=default_logging_level, force=True)

for name in ("httpx", "httpcore", "openai"):
    logger = logging.getLogger(name)
    logger.setLevel(default_logging_level)
    logger.disabled = True



print("✓ Request logging enabled - will show HTTP details below")



tenant = 'public'
environment = ''
base_url = f'https://{tenant}{environment}.tapis.io' 

# Enter Tapis Username. Example: trainingXX
tapis_username = input('Username: ')
# Enter Tapis password. Example: trainingXX
tapis_password = getpass.getpass(prompt='Password: ', stream=None)


#Create python Tapis client for user
tapisClient = Tapis(base_url= base_url, username=tapis_username, password=tapis_password)
# *** Tapis v3: Call to Tokens API
tapisClient.get_tokens()
# Print Tapis v3 token
tapisClient.access_token



✓ Request logging enabled - will show HTTP details below


In [ ]:
tacc_allocation = "MY_PROJECT_ID"  # Ensure this variable is defined

nairrJobDefJSON = f"""
{{
  "name": "tap_flexserv_vista_test",
  "appId": "FlexServ-1.4.0",
  "appVersion": "1.4.0",
  "execSystemId": "vista-tapis-wzhang217",
  "tenant": "public",
  "execSystemLogicalQueue": "gh-dev",
  "maxMinutes": 60,
  "parameterSet": {{
    "appArgs": [
      {{
        "arg": "--flexserv-port 8000",
        "name": "flexServPort"
      }},
      {{
        "arg": "--model-name Qwen/Qwen2.5-Coder-0.5B",
        "name": "modelName"
      }},
      {{
        "arg": "--enable-https",
        "name": "enableHttps"
      }},
      {{
        "arg": "--device auto",
        "name": "device"
      }},
      {{
        "arg": "--dtype bfloat16",
        "name": "dtype"
      }},
      {{
        "arg": "--attn-implementation sdpa",
        "name": "attnImplementation"
      }},
      {{
        "arg": "--model-timeout 86400",
        "name": "modelTimeout"
      }},
      {{
        "arg": "--quantization none",
        "name": "quantization"
      }},
      {{
        "arg": "--is-distributed 0",
        "name": "isDistributed"
      }}
    ],
    "envVariables": [
      {{
        "key": "PRI_MODEL_HOST",
        "value": "HOST_EVAL($SCRATCH)/flexserv/models"
      }},
      {{
        "key": "PUB_MODEL_HOST",
        "value": "/work/projects/aci/cic/models"
      }}
    ],
    "schedulerOptions": [
      {{
        "arg": "--tapis-profile tacc-apptainer",
        "name": "TACC Scheduler Profile"
      }},
      {{
        "name": "TACC Allocation",
        "description": "The TACC Allocation associated with this job execution",
        "include": true,
        "arg": "-A {tacc_allocation}"
      }}
    ]
  }},
  "visible": true
}}
"""


nairrJobDef = json.loads(nairrJobDefJSON)
submitted_job = tapisClient.jobs.submitJob(**nairrJobDef)
print(f"Submitted job with UUID: {submitted_job.uuid}")

Submitted job with UUID: f1f7f123-7011-4591-9dd4-df8e854f822e-007


In [5]:
submitted_job = tapisClient.jobs.getJob(jobUuid=submitted_job.uuid)
print(submitted_job.uuid)
print(submitted_job.name)
print(submitted_job.status)
print(submitted_job.execSystemId)
print(submitted_job.execSystemOutputDir)
last_status = submitted_job.status
while True:
    submitted_job = tapisClient.jobs.getJob(jobUuid=submitted_job.uuid)
    if submitted_job.status != last_status:
        print(f"Job status changed: {submitted_job.status}")
        last_status = submitted_job.status
    if submitted_job.status in ["RUNNING", "FAILED", "FINISHED", "CANCELED"]:
        break
    time.sleep(10)

f1f7f123-7011-4591-9dd4-df8e854f822e-007
tap_flexserv_vista_test
STAGING_JOB
vista-tapis-wzhang217
/scratch/10899/wzhang217/tapis/f1f7f123-7011-4591-9dd4-df8e854f822e-007/output
Job status changed: RUNNING


In [7]:
tapisClient.jobs.getJobHistory(jobUuid=submitted_job.uuid)

[
 created: 2026-03-24T19:30:26.668859Z
 description: {"newJobStatus":"PENDING","oldJobStatus":null,"jobName":"tap_flexserv_vista_test","jobUuid":"f1f7f123-7011-4591-9dd4-df8e854f822e-007","jobOwner":"wzhang217","message":"The job has transitioned to a new status: PENDING."}
 event: JOB_NEW_STATUS
 eventDetail: PENDING
 transferSummary: 
 
 transferTaskUuid: None,
 
 created: 2026-03-24T19:30:26.932784Z
 description: {"newJobStatus":"PROCESSING_INPUTS","oldJobStatus":"PENDING","jobName":"tap_flexserv_vista_test","jobUuid":"f1f7f123-7011-4591-9dd4-df8e854f822e-007","jobOwner":"wzhang217","message":"The job has transitioned to a new status: PROCESSING_INPUTS. The previous job status was PENDING."}
 event: JOB_NEW_STATUS
 eventDetail: PROCESSING_INPUTS
 transferSummary: 
 
 transferTaskUuid: None,
 
 created: 2026-03-24T19:30:27.299800Z
 description: {"newJobStatus":"STAGING_INPUTS","oldJobStatus":"PROCESSING_INPUTS","jobName":"tap_flexserv_vista_test","jobUuid":"f1f7f123-7011-4591-9dd4-d

In [8]:
accessInfoMarker = "ACCESS INFORMATION"
smokeTestMarker = "[smoke_test]"
serverReadyMarker = "Server ready to accept requests"

page=1
while page <= 5:
    output_page = tapisClient.files.getContents(systemId=submitted_job.execSystemId, path=submitted_job.execSystemOutputDir+"/tapisjob.out", more=page)
    if not output_page:
        break
    if isinstance(output_page, (bytes, bytearray)):
        log_text = output_page.decode("utf-8", errors="replace")
    else:
        log_text = str(output_page)
    if serverReadyMarker in log_text:
        # print(log_text, end="", flush=True)
        break
    time.sleep(5)
    page+=1

print("Useful Information from Job Logs:")
print("======================================================================")
lines = log_text.splitlines()

for i, line in enumerate(lines):
    if accessInfoMarker in line:
        print(f"{line}")
        end = min(i + 1 + 10, len(lines))
        for j in range(i + 1, end):
            print(f"{lines[j]}")
            if lines[j].startswith("FlexServ address:"):
                # extract the address and TAP token for later use
                flexserv_address = lines[j].split(" ")[2].strip()
                tap_token = lines[j].split(" ")[6].strip()
                print(f"Extracted FlexServ address: {flexserv_address}, tap_token: {tap_token}")
    
    if smokeTestMarker in line:
        print(f"{line}")
    if serverReadyMarker in line:
        print(f"{line}")
        break

Useful Information from Job Logs:
ACCESS INFORMATION
FlexServ address: https://vista.tacc.utexas.edu:60626  TAP token: c6480e76f7fa5b27bd273a45fa23bfa857c7998acaec9d4272fdb39e90f15e52
Extracted FlexServ address: https://vista.tacc.utexas.edu:60626, tap_token: c6480e76f7fa5b27bd273a45fa23bfa857c7998acaec9d4272fdb39e90f15e52
Anyone on the TACC network or with VPN can access this URL directly!
No additional SSH tunneling required from client machines.

Starting FlexServ in Apptainer container...
Service will be available on port 8000
VENV_PATH=/work/10899/wzhang217/vista/venvs
Launching FlexServ container in SINGLE-NODE mode...
2026-03-24 14:31:36 - __main__ - INFO - Server ready to accept requests


In [ ]:
TAP_TOKEN=tap_token
# Configuration from environment variables
api_key = TAP_TOKEN
base_url = f"{flexserv_address}/v1"

print(f"API Key: {api_key}")
print(f"Base URL: {base_url}")

# Create a custom httpx client with SSL verification disabled
http_client = httpx.Client(verify=False)

openai_client = OpenAI(
    api_key=api_key,
    base_url=base_url,
    http_client=http_client,
)

print("✓ OpenAI client initialized")

@dataclass
class FlexServChatSession:
    model: str
    system_prompt: str
    summary_model: str | None = None
    keep_last_pairs: int = 4               # keep recent raw turns
    summarize_after_pairs: int = 8         # trigger summary when convo grows
    messages: List[Dict[str, str]] = field(default_factory=list)
    turn_log: List[Dict[str, Any]] = field(default_factory=list)
    summary_text: str | None = None

    def __post_init__(self):
        self.messages = [{"role": "system", "content": self.system_prompt}]

    def ask(
        self,
        user_text: str,
        *,
        max_tokens: int = 1000,
        temperature: float = 0.1,
        top_p: float = 1.0,
        seed: int = 42,
        stream: bool = True,
    ) -> str:
        # 1) append user turn
        self.messages.append({"role": "user", "content": user_text})

        # NOTE: remove n=1 for transformers serve; it can be rejected as unused.
        resp = openai_client.chat.completions.create(
            model=self.model,
            messages=self.messages,
            stream=stream,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            seed=seed,
        )

        # 2) stream print + accumulate assistant text
        assistant_text_parts: List[str] = []
        if stream:
            for chunk in resp:
                delta = chunk.choices[0].delta.content
                if delta:
                    print(delta, end="", flush=True)
                    assistant_text_parts.append(delta)
            print("\n\n✓ Streaming complete")
            assistant_text = "".join(assistant_text_parts).strip()
        else:
            assistant_text = resp.choices[0].message.content or ""
            print(assistant_text)
            print("\n\n✓ Complete")

        # 3) append assistant turn (critical for true multi-turn continuity)
        self.messages.append({"role": "assistant", "content": assistant_text})

        # 4) audit log per turn
        self.turn_log.append(
            {
                "ts": time.time(),
                "user": user_text,
                "assistant": assistant_text,
                "model": self.model,
            }
        )

        # 5) summarize old context when long
        self._maybe_summarize_context()

        return assistant_text

    def _maybe_summarize_context(self):
        # count only user+assistant turns (exclude system)
        non_system = [m for m in self.messages if m["role"] != "system"]
        pair_count = len(non_system) // 2
        if pair_count < self.summarize_after_pairs:
            return

        # keep recent raw turns
        keep_msgs = self.keep_last_pairs * 2
        old_part = non_system[:-keep_msgs]
        recent_part = non_system[-keep_msgs:]

        if not old_part:
            return

        summarizer_model = self.summary_model or self.model
        summarize_req = [
            {
                "role": "system",
                "content": (
                    "Summarize prior conversation for a coding assistant. "
                    "Keep: requirements, constraints, decisions, unresolved tasks. "
                    "Output concise bullet points."
                ),
            },
            {"role": "user", "content": json.dumps(old_part, ensure_ascii=False)},
        ]

        s = openai_client.chat.completions.create(
            model=summarizer_model,
            messages=summarize_req,
            stream=False,
            max_tokens=400,
            temperature=0.0,
            top_p=1.0,
        )

        summary = (s.choices[0].message.content or "").strip()
        self.summary_text = summary

        # rebuild context: original system + summary system + recent raw turns
        self.messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "system", "content": f"Conversation summary so far:\n{summary}"},
            *recent_part,
        ]


API Key: c6480e76f7fa5b27bd273a45fa23bfa857c7998acaec9d4272fdb39e90f15e52
Base URL: https://vista.tacc.utexas.edu:60626/v1
✓ OpenAI client initialized


In [11]:
# model_to_test = "Qwen/Qwen2.5-Coder-0.5B"
# model_to_test = "Qwen/Qwen2.5-Coder-7B-Instruct"
model_to_test = "Qwen/Qwen3.5-0.8B"
system_prompt="You are a coding assistant that writes clear, correct, beginner-friendly Python code for machine learning tasks."

session = FlexServChatSession(
    model=model_to_test,
    system_prompt=system_prompt,
)


prompt_message = (
                "Write Python code that reads all images from an input directory stored in the variable IMAGE_DIR.\n\n"
                "TASK DESCRIPTION:\n"
                "- This is an IMAGE-LEVEL BINARY CLASSIFICATION task implemented using an object detection model.\n"
                "- The goal is to determine whether an image contains an animal or not.\n\n"
                "DATASET STRUCTURE:\n"
                "- IMAGE_DIR contains two subdirectories:\n"
                "  * animal     → images that contain at least one animal\n"
                "  * no_animal  → images that contain no animals\n"
                "- Ground-truth labels come ONLY from the directory name.\n\n"
                "MODEL REQUIREMENTS:\n"
                "- Use ONLY a pretrained Ultralytics YOLO detection model (no training or fine-tuning).\n"
                "- Load the model using the Ultralytics YOLO API.\n"
                "- Assume YOLO detects animals using a single class ID (animal).\n\n"
                "DETECTION LOGIC (IMPORTANT):\n"
                "- Run object detection on each image.\n"
                "- If the model produces AT LEAST ONE detection of class `animal` with confidence >= 0.5:\n"
                "    → The image-level prediction is `animal`.\n"
                "- If the model produces NO detections with confidence >= 0.5:\n"
                "    → The image-level prediction is `no_animal`.\n"
                "- YOLO never explicitly predicts `no_animal`; it is inferred from the absence of detections.\n\n"
                "EVALUATION METRICS:\n"
                "- Compare the image-level prediction with the ground-truth label.\n"
                "- Count:\n"
                "  * True Positives  (animal image, animal detected)\n"
                "  * True Negatives  (no_animal image, no detections)\n"
                "  * False Positives (no_animal image, detection present)\n"
                "  * False Negatives (animal image, no detections)\n\n"
                "ACCURACY DEFINITION:\n"
                "- Overall accuracy = (True Positives + True Negatives) / Total Images\n\n"
                "OUTPUT REQUIREMENTS:\n"
                "- Print for each image: filename, detected classes, and confidence scores.\n"
                "- At the end, print:\n"
                "  * Total images processed\n"
                "  * Total animal images (ground truth)\n"
                "  * True Positives\n"
                "  * True Negatives\n"
                "  * False Positives\n"
                "  * False Negatives\n"
                "  * Overall detection accuracy\n\n"
                "CODING REQUIREMENTS:\n"
                "- Store the directory path in IMAGE_DIR.\n"
                "- Read only .jpg, .jpeg, and .png files.\n"
                "- Include all required Python packages.\n"
                "- Include clear comments explaining each step.\n"
                "- Keep the code simple, readable, and beginner-friendly.\n\n"
                "After the code, briefly explain how the program works in plain English."
            )

session.ask(prompt_message)

Here is a complete, beginner-friendly Python script to solve this task.

### How the Code Works (Plain English)

1.  **Imports**: We import the necessary libraries (`os`, `yolo`, `yolo_utils`, `torchvision`) and the `torch` library for the model.
2.  **Directory Setup**: We define the `IMAGE_DIR` variable.
3.  **Model Loading**: We load the YOLOv8 model using the `yolo_utils` library. This is the standard way to load models in Python without writing a custom model.
4.  **Image Processing**: We iterate through every image in the directory.
    *   We check if the file extension is `.jpg`, `.jpeg`, or `.png`.
    *   We load the image into a tensor.
    *   We run the YOLO model on it.
5.  **Prediction Logic**:
    *   We check the model's output. If it has a class named `animal` with a confidence score of 0.5 or higher, we mark the image as `animal`.
    *   If no animal is detected, we mark it as `no_animal`.
6.  **Evaluation**: We calculate the counts of True Positives, True Negatives

'Here is a complete, beginner-friendly Python script to solve this task.\n\n### How the Code Works (Plain English)\n\n1.  **Imports**: We import the necessary libraries (`os`, `yolo`, `yolo_utils`, `torchvision`) and the `torch` library for the model.\n2.  **Directory Setup**: We define the `IMAGE_DIR` variable.\n3.  **Model Loading**: We load the YOLOv8 model using the `yolo_utils` library. This is the standard way to load models in Python without writing a custom model.\n4.  **Image Processing**: We iterate through every image in the directory.\n    *   We check if the file extension is `.jpg`, `.jpeg`, or `.png`.\n    *   We load the image into a tensor.\n    *   We run the YOLO model on it.\n5.  **Prediction Logic**:\n    *   We check the model\'s output. If it has a class named `animal` with a confidence score of 0.5 or higher, we mark the image as `animal`.\n    *   If no animal is detected, we mark it as `no_animal`.\n6.  **Evaluation**: We calculate the counts of True Positives

In [12]:
added_prompt = """
Can you also add metrics like precision, recall, and F1-score to the output? Please modify the code to include these metrics in the final printout, along with the overall detection accuracy. Keep the code clear and beginner-friendly. Just show me what can be appended to what you have already generated.
"""

session.ask(added_prompt)


Here is the complete, modified code. I have added the metrics (Precision, Recall, F1-Score) and the overall accuracy calculation.

### Key Changes Made:
1.  **Added Metrics**: Added a new variable `metrics` to store the calculated values.
2.  **Added Accuracy Calculation**: Added a function to calculate the overall accuracy.
3.  **Formatted Output**: Added a specific section at the end to print the summary statistics (Total, TP, TN, FP, FN, Accuracy).
4.  **Cleaner Code**: Removed unnecessary comments and made the logic flow more direct.

### Complete Code

```python
import os
import yolo_utils
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import cv2
import json

# ==========================================
# 1. CONFIGURATION
# ==========================================

# Define the input directory
IMAGE_DIR = "your_image_directory"

# Define the model path (replace with your actual model file path)
# yolo_utils.load_model() is the standard way 

'Here is the complete, modified code. I have added the metrics (Precision, Recall, F1-Score) and the overall accuracy calculation.\n\n### Key Changes Made:\n1.  **Added Metrics**: Added a new variable `metrics` to store the calculated values.\n2.  **Added Accuracy Calculation**: Added a function to calculate the overall accuracy.\n3.  **Formatted Output**: Added a specific section at the end to print the summary statistics (Total, TP, TN, FP, FN, Accuracy).\n4.  **Cleaner Code**: Removed unnecessary comments and made the logic flow more direct.\n\n### Complete Code\n\n```python\nimport os\nimport yolo_utils\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nimport numpy as np\nimport cv2\nimport json\n\n# ==========================================\n# 1. CONFIGURATION\n# ==========================================\n\n# Define the input directory\nIMAGE_DIR = "your_image_directory"\n\n# Define the model path (replace with your actual model file path)\n# yolo_utils.load

In [13]:
added_prompt_2 = """
One more thing - can you also add a confusion matrix to the output? Please modify the code to include a confusion matrix in the final printout, along with the precision, recall, F1-score, and overall detection accuracy. The confusion matrix should show the counts of true positives, true negatives, false positives, and false negatives in a clear format. Keep the code clear and beginner-friendly. Just show me what can be appended to what you have already generated.
"""
session.ask(added_prompt_2)

Here is the complete, modified code. I have added the metrics (Precision, Recall, F1-Score) and the confusion matrix to the output, formatted clearly for beginners.

### Key Changes Made:
1.  **Added Metrics**: Calculated Precision, Recall, and F1-Score.
2.  **Added Accuracy**: Calculated the overall accuracy.
3.  **Confusion Matrix**: Added a table showing the confusion matrix (TP, TN, FP, FN).
4.  **Formatted Output**: Added a summary section at the end with the counts and accuracy.

### Python Code

```python
import os
import yolo_utils
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import json

# ==========================================
# 1. CONFIGURATION
# ==========================================

# Define the input directory
IMAGE_DIR = "your_image_directory"

# Define the model path (replace with your actual model file path)
MODEL_PATH = "yolo/v8/yolo_v8.pt"

# ==========================================
# 2. DATA PROCESSING
# ==========

'Here is the complete, modified code. I have added the metrics (Precision, Recall, F1-Score) and the confusion matrix to the output, formatted clearly for beginners.\n\n### Key Changes Made:\n1.  **Added Metrics**: Calculated Precision, Recall, and F1-Score.\n2.  **Added Accuracy**: Calculated the overall accuracy.\n3.  **Confusion Matrix**: Added a table showing the confusion matrix (TP, TN, FP, FN).\n4.  **Formatted Output**: Added a summary section at the end with the counts and accuracy.\n\n### Python Code\n\n```python\nimport os\nimport yolo_utils\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nimport numpy as np\nimport json\n\n# ==========================================\n# 1. CONFIGURATION\n# ==========================================\n\n# Define the input directory\nIMAGE_DIR = "your_image_directory"\n\n# Define the model path (replace with your actual model file path)\nMODEL_PATH = "yolo/v8/yolo_v8.pt"\n\n# ==========================================\n# 

In [14]:
added_prompt_3 = """
Also, since our code will be running in a jupyter notebook on an HPC, so it is better to use some variables for the dataset paths instead of hardcoded sample string. Also, if every possible, please optimize the code to make sure it can run efficiently on an HPC environment. For example, you can consider using batch processing for the images instead of processing them one by one, and you can also consider using multiprocessing or parallel processing to speed up the inference if the dataset is large. Please modify the code to include these optimizations while keeping it clear and beginner-friendly. you can also show me another version that uses mpi4py for parallel processing on HPC.
"""
session.ask(added_prompt_3, max_tokens=3000)

Here is the complete, optimized, and beginner-friendly Python code.

### Key Improvements Made:
1.  **Variable Naming**: Replaced hardcoded strings with `IMAGE_DIR`, `MODEL_PATH`, and `NUM_WORKERS` to make the code easier to read and debug.
2.  **Batch Processing**: Instead of processing one image at a time, the code now uses `itertools.chain` to process all images in a single batch. This is significantly faster for large datasets.
3.  **Parallel Processing**: Added `multiprocessing` to run the inference in the background. This is crucial for HPC environments where a single CPU might be too slow to process a large dataset.
4.  **Metrics Calculation**: Added logic to calculate Precision, Recall, and F1-Score dynamically based on the actual predictions.
5.  **Confusion Matrix**: Added a specific section to print the confusion matrix (TP, TN, FP, FN) alongside the accuracy.

### Python Code

```python
import os
import yolo_utils
import torch
import torch.nn as nn
import torch.nn.functiona

'Here is the complete, optimized, and beginner-friendly Python code.\n\n### Key Improvements Made:\n1.  **Variable Naming**: Replaced hardcoded strings with `IMAGE_DIR`, `MODEL_PATH`, and `NUM_WORKERS` to make the code easier to read and debug.\n2.  **Batch Processing**: Instead of processing one image at a time, the code now uses `itertools.chain` to process all images in a single batch. This is significantly faster for large datasets.\n3.  **Parallel Processing**: Added `multiprocessing` to run the inference in the background. This is crucial for HPC environments where a single CPU might be too slow to process a large dataset.\n4.  **Metrics Calculation**: Added logic to calculate Precision, Recall, and F1-Score dynamically based on the actual predictions.\n5.  **Confusion Matrix**: Added a specific section to print the confusion matrix (TP, TN, FP, FN) alongside the accuracy.\n\n### Python Code\n\n```python\nimport os\nimport yolo_utils\nimport torch\nimport torch.nn as nn\nimport t

In [15]:
summarization_prompt = """
Please continue from where you left off, and provide a key summary of this tutorial and take away message.
"""
session.ask(summarization_prompt, max_tokens=3000)

Here is the complete, ready-to-run Python script. It includes the logic to read the dataset, process the images, and generate the requested metrics.

### Key Features of this Code:
1.  **Batch Processing**: It processes all images in the directory at once, which is much faster than looping through them one by one.
2.  **Confusion Matrix**: It calculates the confusion matrix (TP, TN, FP, FN) and calculates the **F1-Score** (the harmonic mean of Precision and Recall).
3.  **Metrics Output**: It prints the specific metrics you requested (Precision, Recall, F1-Score, Accuracy) and the confusion matrix.

### Prerequisites
Make sure you have the following libraries installed:
```bash
pip install torch torchvision torchaudio
```

### Python Script

```python
import os
import yolo_utils
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import json
from itertools import chain

# ==========================================
# 1. CONFIGURATION
# =================

'Here is the complete, ready-to-run Python script. It includes the logic to read the dataset, process the images, and generate the requested metrics.\n\n### Key Features of this Code:\n1.  **Batch Processing**: It processes all images in the directory at once, which is much faster than looping through them one by one.\n2.  **Confusion Matrix**: It calculates the confusion matrix (TP, TN, FP, FN) and calculates the **F1-Score** (the harmonic mean of Precision and Recall).\n3.  **Metrics Output**: It prints the specific metrics you requested (Precision, Recall, F1-Score, Accuracy) and the confusion matrix.\n\n### Prerequisites\nMake sure you have the following libraries installed:\n```bash\npip install torch torchvision torchaudio\n```\n\n### Python Script\n\n```python\nimport os\nimport yolo_utils\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nimport numpy as np\nimport json\nfrom itertools import chain\n\n# ==========================================\n# 1. CONFIG

In [16]:
tapisClient.jobs.cancelJob(jobUuid=submitted_job.uuid)


message: JOBS_JOB_CANCEL_ACCEPTED Request to cancel job f1f7f123-7011-4591-9dd4-df8e854f822e-007 has been accepted. 